# Notebook 08 — Off-Platform Signals & Bayesian Prevalence Update

On-platform signals (classifier scores, engagement metrics, enforcement history) capture
what is detectable within the platform's own data. But harmful actors often leave
traces across the broader internet before — or instead of — appearing on-platform.

**Off-platform signals include:**
- NCMEC CyberTipline hashes (CSAM)
- GIFCT hash-sharing database (terrorist content)
- Google SafeBrowsing URL reputation
- External threat intelligence feeds (leaked credential lists, known bot IPs)
- Cross-platform ban signals (actor banned on platform A → elevated prior on B)
- Academic/OSINT reports (e.g., Stanford Internet Observatory, DFRLab)

## What we build
1. Bayesian update framework: off-platform signal → posterior prevalence estimate
2. Sensitivity analysis: how much does signal strength affect posterior?
3. Signal reliability modeling: accounting for off-platform false positive rates
4. Combined estimator: weighted combination of on-platform and off-platform evidence
5. Decision threshold analysis: when is off-platform evidence sufficient for action?


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

from src.prevalence import PrevalenceEstimator

sns.set_theme(style='whitegrid', palette='muted')
rng = np.random.default_rng(42)

estimator = PrevalenceEstimator(confidence_level=0.95)

## 1. Bayesian update framework

Let π = prevalence of harmful content in a corpus segment.  
Let S = off-platform signal (binary: signal present or absent).

Bayes' theorem gives the posterior:

$$P(\text{harmful} \mid S) = \frac{P(S \mid \text{harmful}) \cdot P(\text{harmful})}{P(S)}$$

At the segment level:

$$\hat{\pi}_{\text{posterior}} = \frac{\text{TPR}_{\text{off}} \cdot \hat{\pi}_{\text{prior}}}{\text{TPR}_{\text{off}} \cdot \hat{\pi}_{\text{prior}} + \text{FPR}_{\text{off}} \cdot (1 - \hat{\pi}_{\text{prior}})}$$

where:
- π_prior = on-platform prevalence estimate
- TPR_off = true positive rate of the off-platform signal for this harm type
- FPR_off = false positive rate of the off-platform signal


In [ ]:
def bayesian_update(
    pi_prior: float,
    tpr_off: float,
    fpr_off: float,
    signal_present: bool = True,
) -> float:
    """Bayesian posterior prevalence after observing an off-platform signal.

    Parameters
    ----------
    pi_prior       : on-platform prevalence estimate (prior)
    tpr_off        : P(signal=1 | truly harmful) for the off-platform source
    fpr_off        : P(signal=1 | not harmful) for the off-platform source
    signal_present : whether the off-platform signal fired (True) or not (False)

    Returns
    -------
    float : posterior prevalence estimate
    """
    if signal_present:
        # P(harmful | signal=1)
        numerator   = tpr_off * pi_prior
        denominator = tpr_off * pi_prior + fpr_off * (1 - pi_prior)
    else:
        # P(harmful | signal=0) — signal absence also updates belief
        numerator   = (1 - tpr_off) * pi_prior
        denominator = (1 - tpr_off) * pi_prior + (1 - fpr_off) * (1 - pi_prior)

    return numerator / denominator if denominator > 0 else pi_prior


# Example: NCMEC hash match (very high precision, high recall for CSAM)
pi_prior_csam = 0.001  # on-platform estimate: 0.1% of media is CSAM
tpr_ncmec     = 0.95   # NCMEC hash match is highly sensitive
fpr_ncmec     = 0.001  # extremely low FPR (cryptographic hashes)

pi_post_match   = bayesian_update(pi_prior_csam, tpr_ncmec, fpr_ncmec, signal_present=True)
pi_post_nomatch = bayesian_update(pi_prior_csam, tpr_ncmec, fpr_ncmec, signal_present=False)

print("NCMEC Hash Signal Example (CSAM)")
print(f"  Prior prevalence:              {pi_prior_csam:.4%}")
print(f"  Posterior (signal present):    {pi_post_match:.4%}  [{pi_post_match/pi_prior_csam:.0f}x lift]")
print(f"  Posterior (signal absent):     {pi_post_nomatch:.4%}")
print()

# Example: Cross-platform ban signal (lower precision)
pi_prior_abuse = 0.005
tpr_ban        = 0.60
fpr_ban        = 0.02

pi_post_ban   = bayesian_update(pi_prior_abuse, tpr_ban, fpr_ban, signal_present=True)
pi_post_noban = bayesian_update(pi_prior_abuse, tpr_ban, fpr_ban, signal_present=False)

print("Cross-Platform Ban Signal (Platform Abuse)")
print(f"  Prior prevalence:              {pi_prior_abuse:.4%}")
print(f"  Posterior (signal present):    {pi_post_ban:.4%}  [{pi_post_ban/pi_prior_abuse:.1f}x lift]")
print(f"  Posterior (signal absent):     {pi_post_noban:.4%}")

## 2. Sensitivity analysis: FPR dominates at low prevalence

At low base rates, FPR has a disproportionate impact on posterior precision.
Even a 1% FPR can overwhelm a 0.1% prior — the signal is mostly noise.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: posterior vs prior for different FPRs (fixed TPR=0.90)
priors = np.linspace(0.0001, 0.05, 200)
fprs   = [0.001, 0.005, 0.010, 0.020, 0.050]
tpr_fixed = 0.90

for fpr in fprs:
    posts = [bayesian_update(p, tpr_fixed, fpr, True) for p in priors]
    axes[0].plot(priors * 100, [p * 100 for p in posts],
                 label=f'FPR={fpr:.1%}', lw=1.8)

axes[0].plot(priors * 100, priors * 100, 'k--', lw=0.8, label='No update (prior)')
axes[0].set_xlabel('Prior prevalence (%)')
axes[0].set_ylabel('Posterior prevalence (%)')
axes[0].set_title(f'Bayesian Update: TPR={tpr_fixed:.0%}, Signal Present\nSensitivity to Off-Platform FPR')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.4)

# Right: posterior lift (post/prior) vs FPR x prior grid
prior_grid = [0.001, 0.005, 0.010, 0.020]
fpr_grid   = np.linspace(0.001, 0.10, 100)
tpr_plot   = 0.90

for pi_p in prior_grid:
    lifts = [bayesian_update(pi_p, tpr_plot, f, True) / pi_p for f in fpr_grid]
    axes[1].plot(fpr_grid * 100, lifts, lw=2, label=f'prior={pi_p:.1%}')

axes[1].axhline(1.0, ls='--', color='gray', alpha=0.5, label='No lift')
axes[1].set_xlabel('Off-platform FPR (%)')
axes[1].set_ylabel('Posterior / Prior (lift)')
axes[1].set_title(f'Signal Lift vs Off-Platform FPR\n(TPR={tpr_plot:.0%})')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.4)

plt.suptitle('Off-Platform Signal Bayesian Update Analysis', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Multiple off-platform signals: sequential updating

When multiple independent signals are available, update the posterior sequentially.
**Independence assumption:** signals must come from truly independent sources —
two platforms sharing the same hash database are NOT independent.


In [ ]:
def sequential_update(
    pi_prior: float,
    signals: list[tuple[float, float, bool]],  # (tpr, fpr, signal_present)
    signal_names: list[str] | None = None,
) -> pd.DataFrame:
    """Apply multiple independent off-platform signals sequentially.

    Parameters
    ----------
    pi_prior     : initial on-platform prevalence estimate
    signals      : list of (tpr, fpr, signal_present) tuples
    signal_names : optional labels for the signals

    Returns
    -------
    pd.DataFrame : posterior after each signal
    """
    names = signal_names or [f"Signal {i+1}" for i in range(len(signals))]
    rows = [{'signal': 'Prior (on-platform)', 'posterior': pi_prior}]

    current = pi_prior
    for name, (tpr, fpr, present) in zip(names, signals):
        current = bayesian_update(current, tpr, fpr, present)
        rows.append({'signal': name, 'posterior': current})

    return pd.DataFrame(rows)


# Scenario: detecting influence operation content
# On-platform estimate: 0.3% influence op content
pi_prior_io = 0.003

io_signals = [
    # (tpr,  fpr,   present)
    (0.70, 0.03,  True),   # GIFCT coordination hash match
    (0.55, 0.05,  True),   # Cross-platform ban on similar account
    (0.40, 0.08,  True),   # OSINT report matching content themes
    (0.60, 0.04,  False),  # Threat intel IP match: NOT present (negative update)
]

io_names = [
    'GIFCT hash match',
    'Cross-platform ban',
    'OSINT report match',
    'Threat intel IP (absent)',
]

df_seq = sequential_update(pi_prior_io, io_signals, io_names)
print("Sequential Bayesian Update — Influence Operations")
print("=" * 50)
for _, row in df_seq.iterrows():
    lift_str = f"  ({row['posterior']/pi_prior_io:.1f}x prior)" if row['signal'] != 'Prior (on-platform)' else ""
    print(f"  {row['signal']:<30} {row['posterior']:.4%}{lift_str}")

In [ ]:
# Visualization of sequential update
fig, ax = plt.subplots(figsize=(10, 5))

posteriors = df_seq['posterior'].values * 100
labels     = df_seq['signal'].tolist()
colors     = ['#4C72B0'] + ['#DD8452' if io_signals[i][2] else '#55A868'
                              for i in range(len(io_signals))]

bars = ax.barh(range(len(labels)), posteriors, color=colors, edgecolor='white', height=0.6)

ax.axvline(pi_prior_io * 100, ls='--', color='gray', alpha=0.7, label='Prior (on-platform only)')

for i, (bar, val) in enumerate(zip(bars, posteriors)):
    ax.text(val + 0.01, i, f'{val:.3f}%', va='center', fontsize=9)

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Posterior prevalence estimate (%)')
ax.set_title('Sequential Bayesian Update: Off-Platform Signals\nInfluence Operations Example')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4C72B0', label='Prior'),
    Patch(facecolor='#DD8452', label='Positive signal (updates upward)'),
    Patch(facecolor='#55A868', label='Negative signal (updates downward)'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
ax.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.show()

## 4. Decision threshold analysis

When does the posterior estimate reach the threshold for enforcement action?
This depends on the cost asymmetry:
- **Cost of false positive** (over-enforcement): chilling effect, appeals volume, user trust
- **Cost of false negative** (under-enforcement): real harm, regulatory risk, reputational damage

Optimal threshold: act when posterior prevalence > k · FP_cost / (FP_cost + FN_cost)


In [ ]:
# Decision threshold as a function of FP/FN cost ratio and posterior estimate
cost_ratios = np.logspace(-1, 2, 200)  # FP_cost / FN_cost

# Posterior after various numbers of positive signals
pi_base    = 0.003  # 0.3% prior
n_signals  = [0, 1, 2, 3]
posteriors_per_nsig = [pi_base]
pi_current = pi_base
for _ in range(3):
    pi_current = bayesian_update(pi_current, 0.65, 0.04, True)  # typical IO signal
    posteriors_per_nsig.append(pi_current)

fig, ax = plt.subplots(figsize=(10, 5))

for n, pi_post in zip(n_signals, posteriors_per_nsig):
    label = f'After {n} signal{"s" if n!=1 else ""} (π={pi_post:.3%})'
    # Threshold: act if posterior > cost_fn / (cost_fn + cost_fp)
    # i.e., act if cost_ratio (FP/FN) is low enough that we act despite errors
    # For given posterior p, act when FP_cost/FN_cost < p/(1-p)
    threshold = pi_post / (1 - pi_post)  # the break-even cost ratio
    ax.axvline(threshold, lw=1.5, ls='--' if n == 0 else '-',
               label=f'{label} → act if FP/FN < {threshold:.3f}')

ax.set_xscale('log')
ax.set_xlabel('FP cost / FN cost ratio (log scale)')
ax.set_title('Enforcement Decision Threshold vs Off-Platform Evidence Strength')
ax.axvspan(0.001, 0.01, alpha=0.1, color='green', label='Low-cost enforcement (auto-action zone)')
ax.axvspan(0.10, 100, alpha=0.1, color='red', label='High-cost enforcement (human review zone)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.005, 50)

# Remove y-axis (no y variable)
ax.set_yticks([])
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 5. Simulating off-platform signal integration at corpus scale

We simulate a corpus where 30% of bad actors have off-platform signal hits,
and measure how integrating the signal improves prevalence estimate precision.


In [ ]:
n_actors       = 50_000
pi_true        = 0.008    # 0.8% true prevalence
signal_hit_rate_bad  = 0.35  # 35% of bad actors have off-platform signal
signal_hit_rate_good = 0.01  # 1% of good actors have off-platform signal (FPR)
tpr_signal     = 0.90
fpr_signal     = signal_hit_rate_good

# Simulate corpus
true_labels   = rng.binomial(1, pi_true, size=n_actors)
signal_hits   = np.where(
    true_labels == 1,
    rng.binomial(1, signal_hit_rate_bad,  size=n_actors),
    rng.binomial(1, signal_hit_rate_good, size=n_actors),
)

# Segment: actors WITH signal hit vs without
mask_hit    = signal_hits == 1
mask_nohit  = ~mask_hit

pi_segment_hit   = true_labels[mask_hit].mean()   if mask_hit.sum() > 0 else 0
pi_segment_nohit = true_labels[mask_nohit].mean() if mask_nohit.sum() > 0 else 0

# Bayesian posterior for each segment
pi_posterior_hit   = bayesian_update(pi_true, tpr_signal, fpr_signal, True)
pi_posterior_nohit = bayesian_update(pi_true, tpr_signal, fpr_signal, False)

print(f"{'Segment':<30} {'True rate':>10} {'Posterior':>12} {'n actors':>10}")
print("-" * 65)
print(f"{'With off-platform signal':<30} {pi_segment_hit:>10.4%} {pi_posterior_hit:>12.4%} {mask_hit.sum():>10,}")
print(f"{'Without off-platform signal':<30} {pi_segment_nohit:>10.4%} {pi_posterior_nohit:>12.4%} {mask_nohit.sum():>10,}")
print(f"{'Overall corpus':<30} {true_labels.mean():>10.4%} {pi_true:>12.4%} {n_actors:>10,}")

In [ ]:
# Show how segmented sampling + off-platform priors reduces review burden
# Allocate review budget proportional to posterior risk

total_review_budget = 3000

# Flat allocation (ignore signal)
n_hit_flat   = int(total_review_budget * mask_hit.mean())
n_nohit_flat = total_review_budget - n_hit_flat

# Risk-proportional allocation
weight_hit   = pi_posterior_hit   * mask_hit.sum()
weight_nohit = pi_posterior_nohit * mask_nohit.sum()
total_weight = weight_hit + weight_nohit
n_hit_risk   = int(total_review_budget * weight_hit   / total_weight)
n_nohit_risk = total_review_budget - n_hit_risk

# Expected true positives found per strategy
tp_flat = n_hit_flat * pi_segment_hit + n_nohit_flat * pi_segment_nohit
tp_risk = n_hit_risk * pi_segment_hit + n_nohit_risk * pi_segment_nohit

print(f"\nReview budget allocation comparison (total n={total_review_budget:,})")
print(f"{'Strategy':<30} {'n (signal)':<12} {'n (no sig)':<12} {'Expected TPs':>12}")
print("-" * 68)
print(f"{'Flat (ignore signal)':<30} {n_hit_flat:<12,} {n_nohit_flat:<12,} {tp_flat:>12.1f}")
print(f"{'Risk-proportional':<30} {n_hit_risk:<12,} {n_nohit_risk:<12,} {tp_risk:>12.1f}")
print(f"\nTP lift from signal integration: {tp_risk/tp_flat:.2f}x")

## 6. Key findings and operational guidance

| Signal type | Typical TPR | Typical FPR | Useful for |
|---|---|---|---|
| NCMEC hash (CSAM) | 0.95 | <0.001 | Immediate auto-action |
| GIFCT hash (terrorism) | 0.85 | 0.005 | High-confidence routing |
| Cross-platform ban | 0.60 | 0.02 | Elevated prior for review |
| OSINT report | 0.45 | 0.05 | Corpus segment flagging |
| IP reputation feed | 0.35 | 0.08 | Actor risk score boost |

**Key lessons:**

1. **FPR dominates at low prevalence.** A 5% FPR against a 0.1% prior means 98% of
   "flagged" items are false alarms. Signal precision must be evaluated in context of
   the specific harm base rate.

2. **Sequential independence is critical.** Platforms sharing the same underlying
   database cannot provide independent updates — this is a common mistake in
   multi-signal frameworks.

3. **Negative signals also update.** The absence of a strong off-platform signal
   (especially from high-recall sources like NCMEC) provides meaningful downward
   probability pressure.

4. **Segment-level Bayesian priors improve review allocation.** Allocating review
   budget proportional to posterior risk (rather than uniform) can yield 1.3-2x
   more true positives discovered per review dollar.

5. **Document signal reliability.** Off-platform sources have their own bias and
   coverage limitations. TPR/FPR estimates should come from labeled validation sets,
   not assumed.
